In [ ]:

#!/usr/bin/env python3
"""
Script para descargar los informes post partido de MediaCoach (Versión Local)

Pasos:
1 - Obtener credenciales
2 - Definición de funciones
3 - Interfaz de usuario y ejecución del código

Importante:
* La estructura de las carpetas donde se guardan los archivos debe ser la siguiente: 
  ./VCF_Mediacoach_Data/Temporada_yy_yy/Competicion (Ej: ./VCF_Mediacoach_Data/Temporada_24_25/La_Liga)
* Dentro de la carpeta de cada competición, para cada temporada, tiene que haber un archivo csv con los ids de los partidos ya procesados.
  ./VCF_Mediacoach_Data/Temporada_yy_yy/ids_procesados_competicion.csv (Ej: ./VCF_Mediacoach_Data/Temporada_23_24/ids_procesados_La_Liga_2.csv)
"""

import requests
import subprocess
import json
import zipfile
import csv
import os
import io
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path

In [ ]:
# ==================== 1 - OBTENER CREDENCIALES ====================

def obtener_credenciales():
    """Obtiene las credenciales de acceso a la API de MediaCoach"""
    
    # Establece los parámetros necesarios para la solicitud
    client_id = '58191b89-cee4-11ed-a09d-ee50c5eb4bb5'
    scope = 'b2bapiclub-api'
    grant_type = 'password'
    username = 'b2bvillarealcf@mediacoach.es'
    password = 'r728-FHj3RE!'

    # URL a la que se hará la solicitud de token
    token_url = 'https://id.mediacoach.es/connect/token'

    # Datos que se enviarán en la solicitud
    data = {
        'client_id': client_id,
        'scope': scope,
        'grant_type': grant_type,
        'username': username,
        'password': password
    }

    # Encabezados para la solicitud
    headers = {
        'Content-Type': 'application/x-www-form-urlencoded'
    }

    # Realiza la solicitud POST
    response = requests.post(token_url, data=data, headers=headers)

    # Verifica si la solicitud fue exitosa
    if response.status_code == 200:
        # Convierte la respuesta en JSON y obtiene el access_token
        token_response = response.json()
        AccessToken = token_response.get('access_token', '')
        expires_in = token_response.get('expires_in', '')
        token_type = token_response.get('token_type', '')
        scope = token_response.get('scope', '')

        # Imprime el access_token y otros detalles
        print(f'Access Token: {AccessToken}')
        print(f'Expires In: {expires_in}')
        print(f'Token Type: {token_type}')
        print(f'Scope: {scope}')
        
        return AccessToken
    else:
        # Imprime el mensaje de error si la solicitud no fue exitosa
        print(f'Error: {response.status_code}')
        print(response.text)
        return None

# Configuración global
SubscriptionKey = '729f9154234d4ff3bb0a692c6a0510c4'
api_url_base = "https://club-api.mediacoach.es"
AccessToken = obtener_credenciales()

if not AccessToken:
    print("No se pudieron obtener las credenciales. Terminando programa.")
    exit(1)

# Función para hacer consultas curl
credenciales = f"--header 'Ocp-Apim-Subscription-Key: {SubscriptionKey}' --header 'Authorization: Bearer {AccessToken}'"

def ejecutar_curl_comando(comando):
    """Ejecuta un comando curl y devuelve el resultado en JSON"""
    process = subprocess.Popen(comando, stdout=subprocess.PIPE, stderr=subprocess.PIPE, shell=True)
    stdout, stderr = process.communicate()
    if process.returncode != 0:
        print("Error en el comando curl:")
        print(stderr.decode())
        return None
    return json.loads(stdout)

In [ ]:
# ==================== 2 - DEFINICIÓN DE FUNCIONES ====================

def convertir_csv_a_parquet(ruta_csv):
    """Convierte un archivo CSV a formato Parquet y elimina el CSV original"""
    try:
        # Leer el CSV
        df = pd.read_csv(ruta_csv)
        
        # Crear la ruta del archivo parquet
        ruta_parquet = ruta_csv.replace('.csv', '.parquet')
        
        # Guardar como parquet
        df.to_parquet(ruta_parquet, index=False)
        
        # Eliminar el archivo CSV original
        os.remove(ruta_csv)
        
        print(f"Convertido a Parquet: {ruta_parquet}")
        return ruta_parquet
    except Exception as e:
        print(f"Error al convertir {ruta_csv} a Parquet: {e}")
        return ruta_csv

def guardar_id_en_csv(id, temporada="Temporada_24_25", competicion="La_Liga", archivo_ids=""):
    """Guarda los match_id tras leerlos del archivo ids_procesados.csv"""
    nombre_archivo = f'./VCF_Mediacoach_Data/{temporada}/{competicion}/{archivo_ids}'
    
    # Crear directorios si no existen
    os.makedirs(os.path.dirname(nombre_archivo), exist_ok=True)
    
    with open(nombre_archivo, 'a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow([id])

def leer_ids_csv(temporada="Temporada_24_25", competicion="La_Liga", archivo_ids="ids_procesados_La_Liga.csv"):
    """Devuelve los match_id tras leerlos del archivo ids_procesados.csv"""
    nombre_archivo = f'./VCF_Mediacoach_Data/{temporada}/{competicion}/{archivo_ids}'
    print(f"Buscando archivo: {nombre_archivo}")
    
    if not os.path.exists(nombre_archivo):
        print(f"Archivo '{nombre_archivo}' no encontrado. Creando archivo vacío.")
        # Crear directorios si no existen
        os.makedirs(os.path.dirname(nombre_archivo), exist_ok=True)
        # Crear archivo vacío
        with open(nombre_archivo, 'w', newline='') as file:
            pass
        return []
    else:
        with open(nombre_archivo, 'r', newline='') as file:
            reader = csv.reader(file)
            match_id_list = []
            for row in reader:
                if not row:
                    continue
                else:
                    match_id_list.append(row[0])
            return match_id_list

def obtener_ids(max_match_day, season_id, temporada, competition_id, competicion, archivo_ids):
    """Devuelve una lista con los match_ids que se jugaron antes de los max_match_day y que no están en la planilla ids_procesados existente"""
    print(f"Procesando: {temporada}, {competicion}, {archivo_ids}")
    ids_existente = leer_ids_csv(temporada, competicion, archivo_ids)
    matches = ejecutar_curl_comando(f"""
    curl --location '{api_url_base}/Championships/seasons/{season_id}/competitions/{competition_id}/matches' \\
    {credenciales}
    """)

    if not matches:
        return []
    ids_filtrados = [item['id'] for item in matches if int(item['matchDayNumber']) < int(max_match_day)]
    return [id for id in ids_filtrados if id not in ids_existente]

def descargar_y_guardar_archivo(file_url, temporada="Temporada_24_25", competicion="La_Liga", carpeta="Carpeta General", nombre_archivo="Archivo general"):
    """Descarga un archivo a partir de la url (file_url). Una vez descargado lo guarda en la carpeta correspondiente"""
    file_response = requests.get(file_url)
    ruta_completa = f'./VCF_Mediacoach_Data/{temporada}/{competicion}/{carpeta}/{nombre_archivo}'
    
    if file_response.status_code == 200:
        # Crear el directorio si no existe
        os.makedirs(os.path.dirname(ruta_completa), exist_ok=True)
        with open(ruta_completa, 'wb') as file:
            file.write(file_response.content)
        
        # Si es un archivo CSV, convertirlo a Parquet
        if nombre_archivo.endswith('.csv'):
            ruta_parquet = convertir_csv_a_parquet(ruta_completa)
            return ruta_parquet
        
        return ruta_completa
    return False

def descargar_archivos(ids, temporada="Temporada_24_25", competicion="La_Liga", archivo_ids=""):
    """Recibe la lista de ids a descargar y descarga los archivos de mediacoach con esa id.
    Cada Id de partido tiene 8 archivos asociados"""
    nombres_de_archivos = []
    total_archivos = len(ids) * 8  # Suponiendo que cada ID tiene 8 archivos asociados.
    archivos_descargados = 0

    # Recorro lista con ids de los partidos
    for i, id in enumerate(ids):
        # Obtengo diccionario con informes de partido
        asset_data = ejecutar_curl_comando(f"curl --location '{api_url_base}/Assets/{id}' {credenciales}")
        if not asset_data:
            continue

        try:
            # Me quedo con las url que me interesan
            file_urls = [asset_data[0]['url'], asset_data[1]['url'], asset_data[5]['url'], asset_data[9]['url'],
                         asset_data[11]['url'], asset_data[12]['url'], asset_data[13]['url'], asset_data[14]['url']]
        except (IndexError, KeyError) as e:
            print(f"Error al procesar el ID {id}: {e}")
            continue  # Continúa con el siguiente ID en caso de error

        # Asigno a cada url una carpeta y una extensión de acuerdo al archivo a descargar
        carpetas = ['Rendimiento', 'Rendimiento', 'postpartidoequipos', 'postpartidojugador',
                    'maximaexigencia', 'maximaexigencia', 'beyondstats', 'maximaexigenciarevisada']
        extensiones = ['xlsx', 'xlsx', 'csv', 'csv', 'xlsx', 'xlsx', 'xml', 'xml']

        # Creo lista con diccionarios: [{file_url, carpeta, extension}] y la recorro
        for j, (file_url, carpeta, extension) in enumerate(zip(file_urls, carpetas, extensiones)):
            # Descargo el archivo y lo guardo en el directorio local en la carpeta correspondiente
            if file_url:
                archivo_descargado = descargar_y_guardar_archivo(
                    file_url, 
                    temporada=temporada, 
                    competicion=competicion, 
                    carpeta=carpeta, 
                    nombre_archivo=f'archivo_{id}_descargado_{i}_{j}.{extension}'
                )
                if archivo_descargado:
                    nombres_de_archivos.append(f'{carpeta}/{os.path.basename(archivo_descargado)}')
                    archivos_descargados += 1
                    porcentaje_completado = (archivos_descargados / total_archivos) * 100
                    print(f"Progreso: {porcentaje_completado:.2f}% completado ({archivos_descargados}/{total_archivos} archivos)")
        
        # Guardo id del partido en archivo csv
        guardar_id_en_csv(id, temporada, competicion, archivo_ids)

    return nombres_de_archivos

In [ ]:
# ==================== 3 - INTERFAZ DE USUARIO Y EJECUCIÓN DEL CÓDIGO ====================

def main():
    """Función principal que maneja la interfaz de usuario"""
    
    # 1 - Preguntar Temporada
    temporadas = [
        {"nombre": "Temporada 24-25", "id": "3a134240-833f-41dd-c6b0-3d6b87479c15", "input": 0},
        {"nombre": "Temporada 23-24", "id": "3a0bf8ee-7f55-aeb6-fd31-273f2d45aefa", "input": 1}
    ]

    print("Lista de temporadas disponibles:")
    for temporada in temporadas:
        print(f'{temporada["nombre"]}, seleccione {temporada["input"]}')

    while True:
        try:
            input_temporadas = input("Seleccione una temporada (0-1) o pulse 'q' para salir: ")
            if input_temporadas.lower() == 'q':
                print("Saliendo del programa...")
                return
            input_temporadas = int(input_temporadas)
            if input_temporadas in [0, 1]:
                break
            else:
                print("Opción no válida. Seleccione 0 o 1.")
        except ValueError:
            print("Por favor, ingrese un número válido.")

    season_id = temporadas[input_temporadas]["id"]
    season_name = temporadas[input_temporadas]["nombre"]
    temporada = season_name.replace(" ", "_").replace("-", "_")
    print(f"Ha elegido la temporada {season_name}")

    # 2 - Preguntar Competición
    competiciones = [
        {"nombre": "La Liga", "id": "39df9ec8-be91-4be5-1925-4b670a4cbed9", "input": 0},
        {"nombre": "La Liga 2", "id": "39df9ec8-becb-86ea-b5e8-600c1b47968d", "input": 1}
    ]

    print("\nLista de competiciones disponibles:")
    for competicion in competiciones:
        print(f'{competicion["nombre"]}, seleccione {competicion["input"]}')

    while True:
        try:
            input_competiciones = input("Seleccione una competición (0-1) o pulse 'q' para salir: ")
            if input_competiciones.lower() == 'q':
                print("Saliendo del programa...")
                return
            input_competiciones = int(input_competiciones)
            if input_competiciones in [0, 1]:
                break
            else:
                print("Opción no válida. Seleccione 0 o 1.")
        except ValueError:
            print("Por favor, ingrese un número válido.")

    competition_id = competiciones[input_competiciones]["id"]
    competition_name = competiciones[input_competiciones]["nombre"]
    competicion = competition_name.replace(" ", "_").replace("-", "_")
    print(f"Ha elegido la competición {competition_name}")

    archivo_ids = f'ids_procesados_{competicion}.csv'

    # 3 - Preguntar Max match days
    while True:
        try:
            max_match_day = input("Ingrese la cantidad de días de partidos (Ej: 1, 2, 15, etc.) o pulse 'q' para salir: ")
            if max_match_day.lower() == 'q':
                print("Saliendo del programa...")
                return
            max_match_day = int(max_match_day)
            break
        except ValueError:
            print("Por favor, ingrese un número entero válido.")

    print(f"Ha elegido {max_match_day} días de partidos")

    # 4 - Ejecutar función con datos proporcionados
    print("\nObteniendo IDs de partidos...")
    ids = obtener_ids(max_match_day, season_id, temporada, competition_id, competicion, archivo_ids)
    print(f"Se procesarán {len(ids)} partidos")
    
    if ids:
        print("Iniciando descarga de archivos...")
        nombres_de_archivos = descargar_archivos(ids, temporada=temporada, competicion=competicion, archivo_ids=archivo_ids)
        print(f"\n¡Descarga completada! Se descargaron {len(nombres_de_archivos)} archivos.")
        print(f"Los archivos se encuentran en: ./VCF_Mediacoach_Data/{temporada}/{competicion}/")
    else:
        print("No hay nuevos partidos para procesar.")

if __name__ == "__main__":
    # Verificar que las librerías necesarias estén instaladas
    try:
        import pandas as pd
        import pyarrow as pa
        import pyarrow.parquet as pq
    except ImportError as e:
        print("Error: Faltan librerías necesarias.")
        print("Por favor, instala las dependencias con:")
        print("pip install pandas pyarrow requests")
        exit(1)
    
    main()